# 09 · Composer on GCP

> **Module 4 · Cloud Composer · Lecture 4.1**

In this notebook you will:

- Explain what Cloud Composer **is**: managed Apache Airflow running on GKE
- Walk the architecture: Cloud SQL metadata DB, GCS for DAGs, GKE workers, Cloud Storage for logs
- Provision an environment with **`gcloud composer environments create`** — every command in a fenced `bash` block, marked *run in a terminal, not executed here*
- Size an environment and **compute the cost** in a pandas DataFrame with a monthly projection
- Compare **Composer 2 (Celery) vs Composer 3 (KubernetesExecutor)** and the Airflow version each pins
- Tour the **Composer UI** — DAGs, Graph, Calendar, Logs, Admin panes, Config
- Set up **IAM**: the environment's service account and the permissions it needs
- Plan **cleanup and cost control** — deleting an environment, what survives, bucket retention, budget alerts
- **Simulate the environment locally** with a Python dataclass and render the cost table — fully executable

## 1. What Composer is

**Cloud Composer is managed Apache Airflow.** Google Cloud runs the Airflow scheduler, the web server, the workers and the database; you upload DAG files and get a web UI.

That is the whole product in one sentence, and the rest of this notebook is the details that decide whether it is the right tool for you.

### The architecture

Composer 2 and 3 run Airflow on a **Google Kubernetes Engine (GKE)** cluster that Google manages. Around that cluster sit four more managed resources:

```text
        ┌──────────────────────────────────────────────────┐
        │              Cloud Composer (GKE)                │
        │                                                  │
        │  ┌────────────┐   ┌───────────┐   ┌──────────┐  │
        │  │ web server │   │ scheduler │   │ workers  │  │
        │  │  (Airflow  │   │ (Airflow  │   │ (Celery  │  │
        │  │   UI/API)  │   │  scheduler│   │  or K8s  │  │
        │  └─────┬──────┘   └─────┬─────┘   └────┬─────┘  │
        │        │                │              │        │
        │        └────────────────┴──────────────┘        │
        │                         │                        │
        │              ┌──────────▼──────────┐             │
        │              │  Cloud SQL for      │             │
        │              │  PostgreSQL         │             │
        │              │  (metadata DB)      │             │
        │              └─────────────────────┘             │
        └──────────────────────────────────────────────────┘
                        │              │
        ┌───────────────▼──┐        ┌──▼─────────────────────┐
        │  Cloud Storage   │        │  Cloud Storage        │
        │  GCS bucket      │        │  GCS bucket           │
        │  (DAGs)          │        │  (task logs)          │
        │  gs://…/dags/    │        │  gs://…/logs/         │
        └──────────────────┘        └───────────────────────┘
                        │
              ┌─────────▼──────────┐
              │  BigQuery, GCS,    │   ← what your DAGs actually
              │  Cloud SQL, APIs   │      talk to
              └────────────────────┘
```

| Component | What it is | Why it matters |
|---|---|---|
| **Cloud SQL for PostgreSQL** | The Airflow **metadata database** — DAG definitions, run history, XCom, Variables, Connections, pool slots | It is a *real database* with a *real cost*. It is also the single point of failure for the whole environment. |
| **GCS bucket for DAGs** | Where your `.py` files live | Composer watches it and re-parses on change. A DAG file is a source file, not a build artefact. |
| **GKE workers** | Where tasks actually run | Sized by the environment's environment size. This is the compute you pay for. |
| **GCS bucket for logs** | Where task logs are written | Retained for a configurable window — **this is where you look when a task fails**. |
| **Web server** | The Airflow UI and the REST API | Runs in the cluster. |

### What "managed" means, precisely

| You **do** | Google **does** |
|---|---|
| Write DAG files | Run the scheduler, web server, workers, triggerer |
| Choose an environment size | Provision the GKE cluster, node pools, autoscaling |
| Set Variables and Connections | Patch Airflow, patch base images, manage the OS |
| Upload DAGs to the bucket | Restart crashed pods, reschedule, backup the metadata DB |
| Trigger runs, read logs | Keep the Airflow version on a supported release |

What you **still** own: the DAG files, the business logic, the data quality, the cost, and the on-call rotation.

### Composer 2 vs Composer 3

| | **Composer 2** | **Composer 3** |
|---|---|---|
| Executor | **CeleryExecutor** | **KubernetesExecutor** |
| Airflow version pinned | Airflow 2.9.3 | Airflow 2.9.3 (or newer, per release notes) |
| Task execution | A task runs in a **long-lived worker process** | A task runs in a **new Kubernetes Pod** |
| Task startup | ~1 second (the worker is already up) | **~30–60 seconds** (Pod scheduling + image pull) |
| Idle cost | Workers bill continuously | No Pods when idle |
| Per-task resources | Whatever the worker has | Exact requests/limits, per task |
| Per-task images | One image for every task | A different image per task |
| Best for | Most workloads — queries, API calls, moderate Python | Heavy tasks needing their own resources or image |

**The rule:** Composer's default (Celery) has **no startup tax**. Choose the KubernetesExecutor only when you genuinely need per-task resources or per-task images, and accept the 30–60 second cost knowingly.

The crossover is around **7–8 minutes** of task duration. Below it, the Pod tax dominates; above it, isolation and elasticity win.

### What Composer is not

- **It is not a data processing engine.** A `PythonOperator` is one process. If your task takes minutes or holds hundreds of megabytes, it belongs in BigQuery, Dataflow or Dataproc.
- **It is not free.** The environment has a fixed monthly cost regardless of how many jobs you run through it.
- **It is not where your data lives.** Data goes in BigQuery and GCS; the metadata DB holds *control* data, and putting bulk data in XCom is a genuine production mistake.

## 2. Provisioning an environment

> **Everything in this section is a `bash` block. Run it in a terminal — it is NOT executed by this notebook.** The notebook has no credentials, no network, and no `gcloud`.

### Before you start: the three prerequisites

```bash
# 1. Install and authenticate the gcloud CLI.
#    https://cloud.google.com/sdk/docs/install
gcloud --version
gcloud auth login
gcloud config set project YOUR_PROJECT_ID

# 2. Enable the APIs Composer needs.
gcloud services enable \
  composer.googleapis.com \
  compute.googleapis.com \
  sqladmin.googleapis.com \
  storage.googleapis.com \
  cloudresourcemanager.googleapis.com

# 3. Create the two buckets. The DAGs bucket is where you upload your
#    DAG files; the logs bucket is where task logs are written.
#    Composer can create these for you, but creating them yourself lets
#    you set the lifecycle policy and retention you want.
gcloud storage buckets create gs://YOUR_PROJECT_ID-dags \
  --location=REGION \
  --uniform-bucket-level-access

gcloud storage buckets create gs://YOUR_PROJECT_ID-logs \
  --location=REGION \
  --uniform-bucket-level-access
```

### The create command

This is the command that matters. Every flag below has a reason:

```bash
# Create a Composer 2 environment.
# RUN IN A TERMINAL — not executed here.
gcloud composer environments create YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --image-version=composer-2.9.3-airflow-2.9.3 \
  --environment-size=ENVIRONMENT_SIZE \
  --machine-type=YOUR_MACHINE_TYPE \
  --dag-bucket=gs://YOUR_PROJECT_ID-dags \
  --env-vars=KEY1=VALUE1,KEY2=VALUE2 \
  --web-server-network=YOUR_VPC_NETWORK \
  --web-server-subnetwork=YOUR_VPC_SUBNET \
  --oauth-scope=cloud-platform \
  --service-account=YOUR_SERVICE_ACCOUNT \
  --python-version=3.11
```

### What each flag does

| Flag | What it sets | Getting it wrong |
|---|---|---|
| `--location` | The region the environment runs in | The DAGs bucket and the environment should be in the **same region** — otherwise every DAG file read crosses a region boundary, and your data leaves the region it was created in. |
| `--image-version` | The Airflow version and the Composer generation | Pin it explicitly. Never use `latest` in production — a silent Airflow upgrade can break a DAG that parsed fine yesterday. |
| `--environment-size` | `ENVIRONMENT_SIZE_SMALL`, `..._MEDIUM`, `..._LARGE` | See the sizing table below. This is the largest single line item in your bill. |
| `--machine-type` | The GCE machine type for the workers | `n1-standard-2` for most workloads. Bigger machines buy memory, not parallelism. |
| `--env-vars` | Airflow configuration, as `KEY=VALUE` pairs | These are **environment-level config**, not per-run config. Use them for things like `AIRFLOW__CORE__PARALLELISM`. |
| `--dag-bucket` | The GCS bucket holding your DAG files | Point it at the bucket you actually upload to. A wrong bucket is the number-one "my DAG does not appear" cause. |
| `--web-server-network` / `--web-server-subnetwork` | The VPC the Airflow web server sits in | Use a **private** network for anything sensitive. Composer creates a managed VPC if you omit these, which is not what an enterprise deployment wants. |
| `--oauth-scope` | The OAuth scope the workers request | `cloud-platform` gives broad access. Narrow it if your security team requires it — the scope is not the same as the IAM role, but it is a second door. |
| `--service-account` | The identity the workers run as | This is the identity that needs BigQuery and GCS permissions. See the IAM section. |
| `--python-version` | The Python version in the Airflow environment | Must match your DAGs' expectations. A `python_callable` that uses a 3.12-only syntax fails at parse time on 3.11. |

### Airflow configuration through `--env-vars`

Anything in `airflow.cfg` can be set as an environment variable with the `AIRFLOW__<SECTION>__<KEY>` convention:

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --env-vars=\
AIRFLOW__CORE__PARALLELISM=32,\
AIRFLOW__CORE__MAX_ACTIVE_TASKS_PER_DAG=16,\
AIRFLOW__CORE__DEFAULT_TIMEZONE=UTC,\
AIRFLOW__SCHEDULER__DAG_DIR_LIST_INTERVAL=60,\
AIRFLOW__DATABASE__SQL_ALCHEMY_POOL_SIZE=10,\
AIRFLOW__WEBSERVER__EXPOSE_CONFIG=True
```

`AIRFLOW__CORE__DEFAULT_TIMEZONE=UTC` is worth setting explicitly on every environment. Airflow's default timezone has changed between versions, and a timezone difference between the scheduler and your warehouse produces an off-by-one-day bug that is invisible until a monthly report is wrong.

`AIRFLOW__WEBSERVER__EXPOSE_CONFIG=True` shows the effective `airflow.cfg` in the UI — invaluable when debugging, and a reason to keep Variable read access restricted to the people who need it.

### Verifying the environment

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments describe YOUR_ENVIRONMENT_NAME \
  --location=REGION

gcloud composer environments list --location=REGION

# The DAGs that were parsed. An empty list means the DAG files did not
# parse — check the environment's import-error log.
gcloud composer environments run list \
  YOUR_ENVIRONMENT_NAME --location=REGION
```

### Updating and deleting

```bash
# RUN IN A TERMINAL — not executed here.

# Change the size. Workers resize in place; the environment stays up.
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --environment-size=ENVIRONMENT_SIZE_MEDIUM

# Delete. The environment, its GKE cluster, its Cloud SQL instance and
# its workers all go away. The buckets do NOT.
gcloud composer environments delete YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --quiet
```

## 3. Sizing and cost

### The three sizes

Composer 2 environments come in three sizes. The number that matters is the **worker count**, because that is what you pay for.

| Environment size | Workers | Airflow parallelism | Typical use |
|---|---|---|---|
| `ENVIRONMENT_SIZE_SMALL` | **1–2** | 2–8 | A handful of lightweight DAGs, or a dev environment |
| `ENVIRONMENT_SIZE_MEDIUM` | **3–6** | 8–32 | The production default for most teams |
| `ENVIRONMENT_SIZE_LARGE` | **7–15** | 32–128 | Wide fan-outs, many concurrent DAGs |

(The worker count is a range because the workers autoscale within the size's band. The *maximum* is the number your bill can reach.)

### The cost shape — and the number people get wrong

A Composer environment's cost is **mostly fixed**. You pay for the GKE cluster, the node pool, the Cloud SQL instance and the web server **whether or not you run a single job**. The variable part — the BigQuery bytes you scan, the objects you write — is a separate bill entirely.

This is why "run everything through one Composer environment" is usually the right call: the marginal cost of the 200th DAG in an existing environment is close to zero.

### Sizing rule

> **If your DAGs mostly *issue queries*, size the environment small.** A BigQuery query runs server-side; the worker only waits for the answer. **If your DAGs mostly *run Python*, size it for the concurrency you need.**

Let us compute this properly.

In [ ]:
"""The environment cost model, computed.

Every figure below is an *assumption you can change*. That is the point:
the arithmetic is auditable, so you can replace the rates with your own
infrastructure pricing and get your own number.
"""

from dataclasses import dataclass

import pandas as pd

# --------------------------------------------------------------------------
# Pricing assumptions. Replace these with your own rates.
# --------------------------------------------------------------------------

#: USD per GCE n1-standard-2 vCPU-hour, us-central1, on-demand.
#: (4 vCPU, 16 GB — the machine type Composer's workers run on.)
USD_PER_VCPU_HOUR = 0.031611

#: USD per vCPU-hour for GKE cluster management overhead.
USD_PER_VCPU_HOUR_MANAGEMENT = 0.10 / 4  # $0.10/cluster-hour spread over 4 vCPU

#: USD per GB-month for the Cloud SQL PostgreSQL instance backing the
#: metadata database, plus its storage.
USD_PER_GB_MONTH_CLOUD_SQL = 0.20

#: USD per GB-month for Cloud Storage (standard, regional).
USD_PER_GB_MONTH_STORAGE = 0.020

#: USD per GB of logs written per month.
USD_PER_GB_LOGS_WRITTEN = 0.05

#: The fixed monthly cost of the environment's web server, the scheduler
#: and the other always-on components, per environment size.
#: (Composer 2, us-central1, approximate.)
FIXED_MONTHLY_BY_SIZE = {
    "SMALL": 180.0,
    "MEDIUM": 340.0,
    "LARGE": 720.0,
}

#: Hours in a month, for the projection.
HOURS_PER_MONTH = 730


@dataclass(frozen=True)
class EnvironmentSpec:
    """One Composer environment's shape and its cost.

    Attributes:
        name: The environment's name.
        size: The environment size.
        workers_min: The minimum worker count.
        workers_max: The maximum worker count.
        vcpu_per_worker: The vCPUs per worker.
        memory_gb_per_worker: The memory per worker, in GB.
        cloud_sql_gb: The Cloud SQL instance size, in GB.
        logs_gb_month: The logs written per month, in GB.
        dag_storage_gb: The DAG bucket's size, in GB.
    """

    name: str
    size: str
    workers_min: int
    workers_max: int
    vcpu_per_worker: int
    memory_gb_per_worker: int
    cloud_sql_gb: int
    logs_gb_month: float
    dag_storage_gb: float

    @property
    def workers_midpoint(self) -> float:
        """Return the midpoint worker count, used for the projection.

        Returns:
            The average of the minimum and maximum worker counts.
        """
        return (self.workers_min + self.workers_max) / 2


#: The three Composer 2 environment sizes.
ENVIRONMENTS = [
    EnvironmentSpec("composer-dev", "SMALL", 1, 2, 4, 16, 10, 2.0, 0.01),
    EnvironmentSpec("composer-prod", "MEDIUM", 3, 6, 4, 16, 25, 10.0, 0.05),
    EnvironmentSpec("composer-wide", "LARGE", 7, 15, 4, 16, 50, 50.0, 0.10),
]


def worker_hourly_cost(spec: EnvironmentSpec, workers: float) -> float:
    """Return the hourly cost of the workers.

    Args:
        spec: The environment's specification.
        workers: The number of workers.

    Returns:
        The cost per hour, in USD.
    """
    vcpu_hours = workers * spec.vcpu_per_worker
    compute = vcpu_hours * USD_PER_VCPU_HOUR
    management = vcpu_hours * USD_PER_VCPU_HOUR_MANAGEMENT
    return compute + management


def monthly_cost(spec: EnvironmentSpec, *, utilisation: float = 1.0) -> dict:
    """Compute an environment's monthly cost.

    Args:
        spec: The environment's specification.
        utilisation: The fraction of the worker band's midpoint actually
            used. 1.0 means the environment is fully utilised.

    Returns:
        A dict of the cost components, all in USD per month.
    """
    workers = spec.workers_midpoint * utilisation
    worker_cost = worker_hourly_cost(spec, workers) * HOURS_PER_MONTH

    cloud_sql = spec.cloud_sql_gb * USD_PER_GB_MONTH_CLOUD_SQL
    dag_storage = spec.dag_storage_gb * USD_PER_GB_MONTH_STORAGE
    logs = spec.logs_gb_month * USD_PER_GB_LOGS_WRITTEN
    fixed = FIXED_MONTHLY_BY_SIZE[spec.size]

    return {
        "environment": spec.name,
        "size": spec.size,
        "workers": round(workers, 1),
        "worker_compute": round(worker_cost, 2),
        "gke_management": round(
            workers * spec.vcpu_per_worker
            * USD_PER_VCPU_HOUR_MANAGEMENT * HOURS_PER_MONTH, 2),
        "cloud_sql": round(cloud_sql, 2),
        "dag_storage": round(dag_storage, 2),
        "logs": round(logs, 2),
        "fixed_components": round(fixed, 2),
        "total_monthly": round(
            worker_cost + cloud_sql + dag_storage + logs + fixed, 2),
    }


# --------------------------------------------------------------------------
# The cost table
# --------------------------------------------------------------------------
rows = [monthly_cost(spec) for spec in ENVIRONMENTS]
cost_table = pd.DataFrame(rows)

print("Composer 2 environment cost, us-central1, n1-standard-2 workers")
print("=" * 78)
print(cost_table.to_string(index=False))
print()
print(f"assumptions: ${USD_PER_VCPU_HOUR}/vCPU-hour compute, "
      f"${USD_PER_GB_MONTH_CLOUD_SQL}/GB-month Cloud SQL, "
      f"${USD_PER_GB_MONTH_STORAGE}/GB-month storage, "
      f"{HOURS_PER_MONTH} hours/month")
print()

# --------------------------------------------------------------------------
# The utilisation sensitivity — the number that decides your size
# --------------------------------------------------------------------------
print("=" * 78)
print("Monthly cost by utilisation (the workers are the variable part)")
print("=" * 78)

util_rows = []
for spec in ENVIRONMENTS:
    row = {"size": spec.size}
    for utilisation in (0.25, 0.50, 0.75, 1.00):
        row[f"{int(utilisation * 100)}% used"] = monthly_cost(
            spec, utilisation=utilisation)["total_monthly"]
    util_rows.append(row)

util_table = pd.DataFrame(util_rows)
print(util_table.to_string(index=False))
print()
print("Read this carefully: a MEDIUM environment at 25% utilisation costs")
print("almost as much as a MEDIUM at 100%, because the fixed components")
print("(web server, scheduler, Cloud SQL) do not shrink with usage.")
print("That is the argument for ONE environment rather than five.")
print()

# --------------------------------------------------------------------------
# The year projection
# --------------------------------------------------------------------------
print("=" * 78)
print("Twelve-month projection, per environment")
print("=" * 78)

projection = pd.DataFrame(
    [
        {
            "environment": spec.name,
            "size": spec.size,
            "monthly_usd": monthly_cost(spec)["total_monthly"],
            "quarterly_usd": round(monthly_cost(spec)["total_monthly"] * 3, 2),
            "annual_usd": round(monthly_cost(spec)["total_monthly"] * 12, 2),
        }
        for spec in ENVIRONMENTS
    ]
)
projection["daily_usd"] = (projection["monthly_usd"] / 30).round(2)
print(projection.to_string(index=False))
print()

total_annual = projection["annual_usd"].sum()
print(f"All three environments, running together: ${total_annual:,.2f} / year")
print()

# --------------------------------------------------------------------------
# The comparison that matters: environment cost vs job cost
# --------------------------------------------------------------------------
print("=" * 78)
print("ENVIRONMENT COST vs JOB COST")
print("=" * 78)

# A nightly BigQuery aggregation over ~30 GB/day, partitioned.
BQ_PRICE_PER_TB = 5.0
GB_PER_DAY_SCANNED = 30
bq_monthly = GB_PER_DAY_SCANNED * 30 / 1024 * BQ_PRICE_PER_TB

# The same job as a PythonOperator on one worker, running 40 minutes a night.
PYTHON_TASK_HOURS_PER_DAY = 40 / 60
python_monthly = (PYTHON_TASK_HOURS_PER_DAY * 30
                  * 4 * USD_PER_VCPU_HOUR)

comparison = pd.DataFrame([
    {
        "approach": "BigQuery scheduled query (no Composer)",
        "compute_cost_monthly_usd": round(bq_monthly, 2),
        "environment_cost_monthly_usd": 0.0,
        "total_monthly_usd": round(bq_monthly, 2),
        "note": "No environment at all — right answer for a single, dependency-free SQL job",
    },
    {
        "approach": "Composer + PythonOperator (40 min/night)",
        "compute_cost_monthly_usd": round(python_monthly, 2),
        "environment_cost_monthly_usd": 0.0,
        "total_monthly_usd": round(python_monthly, 2),
        "note": "Cheap — but it does not scale, and a retry costs 40 minutes",
    },
    {
        "approach": "Composer (MEDIUM) + BigQuery query",
        "compute_cost_monthly_usd": round(bq_monthly, 2),
        "environment_cost_monthly_usd": 340.0,
        "total_monthly_usd": round(bq_monthly + 340.0, 2),
        "note": "The environment dwarfs the job — size it SMALL",
    },
    {
        "approach": "Composer (MEDIUM) + PythonOperator",
        "compute_cost_monthly_usd": round(python_monthly, 2),
        "environment_cost_monthly_usd": 340.0,
        "total_monthly_usd": round(python_monthly + 340.0, 2),
        "note": "Environment + compute; size for the concurrency you need",
    },
])
print(comparison.to_string(index=False))
print()
print("The design question:")
print()
print('  "If my DAGs mostly ISSUE QUERIES, size the Composer environment')
print('   small. If my DAGs mostly RUN PYTHON, size it for the concurrency')
print('   you need."')
print()
print("And the rule of thumb from the tool-choice chapter:")
print()
print('  "If the task takes more than a few minutes, or holds more than a')
print('   few hundred megabytes, it belongs in a processing engine."')
print()

# --------------------------------------------------------------------------
# Sanity assertions
# --------------------------------------------------------------------------
assert cost_table["total_monthly"].is_monotonic_increasing, \
    "a bigger environment must cost more"
assert cost_table.loc[0, "workers"] < cost_table.loc[1, "workers"]
assert projection["annual_usd"].is_monotonic_increasing

# The fixed components are the same fraction of a SMALL environment's bill
# as they are of a LARGE one's — they do not scale with the worker band.
small = monthly_cost(ENVIRONMENTS[0])
large = monthly_cost(ENVIRONMENTS[2])
small_fixed_share = small["fixed_components"] / small["total_monthly"]
large_fixed_share = large["fixed_components"] / large["total_monthly"]
print(f"fixed-component share, SMALL: {small_fixed_share:.1%}")
print(f"fixed-component share, LARGE: {large_fixed_share:.1%}")
print()
print("PASS: the cost table is internally consistent.")

## 4. A local simulation of the environment

Everything above is a `gcloud` command you cannot run in a notebook. But the *shape* of an environment is just a configuration object — and a configuration object can be modelled, validated and costed in plain Python.

This is worth doing for three reasons:

1. **It is the same exercise as writing Terraform.** If you cannot describe the environment in Python, you cannot describe it in HCL either.
2. **It catches the mistakes that cost money.** A wrong `--location`, an oversized environment, a missing IAM binding — a validator finds them before you create anything.
3. **It makes the cost table reproducible.** The numbers in section 3 come from a dataclass, not from a spreadsheet.

### What a Composer environment actually is

Strip away the cloud API and an environment is:

- a **name** and a **location**
- an **Airflow version** (which pins the Composer generation and the executor)
- a **size** (which determines the worker band)
- a **DAG bucket** and a **logs bucket**
- a **service account** (the identity the workers run as)
- a **network** (the VPC the web server sits in)
- **Airflow configuration** (the `--env-vars`)
- **Variables and Connections** (the environment-level config and the credentials)
- a **lifecycle** (how long it lives, and what happens to the buckets)

Let us model exactly that.

In [ ]:
"""A local model of a Composer environment.

This is the environment as a configuration object: the same fields the
`gcloud composer environments create` command takes, plus the things the
command implies. It is validated, costed, and — crucially — it can be
rendered into the exact command you would run.

No cloud access. No credentials. No network.
"""

from dataclasses import dataclass, field
from typing import Any

# --------------------------------------------------------------------------
# The reference data
# --------------------------------------------------------------------------

#: The Composer generations and the Airflow version each pins.
COMPOSER_GENERATIONS = {
    "composer-2": {
        "executor": "CeleryExecutor",
        "airflow_version": "2.9.3",
        "image_version": "composer-2.9.3-airflow-2.9.3",
        "task_startup_seconds": 1,
    },
    "composer-3": {
        "executor": "KubernetesExecutor",
        "airflow_version": "2.9.3",
        "image_version": "composer-3.0.4-airflow-2.9.3",
        "task_startup_seconds": 45,
    },
}

#: The environment sizes and their worker bands.
ENVIRONMENT_SIZES = {
    "SMALL": {"workers_min": 1, "workers_max": 2},
    "MEDIUM": {"workers_min": 3, "workers_max": 6},
    "LARGE": {"workers_min": 7, "workers_max": 15},
}

#: The IAM roles an environment's service account typically needs.
REQUIRED_ROLES = [
    "roles/composer.worker",
    "roles/bigquery.jobUser",
    "roles/storage.objectAdmin",
    "roles/cloudresourcemanager.projectViewer",
]


@dataclass
class ComposerEnvironment:
    """A Cloud Composer environment, as a configuration object.

    Attributes:
        name: The environment's name.
        location: The region it runs in.
        generation: ``"composer-2"`` or ``"composer-3"``.
        size: The environment size.
        machine_type: The GCE machine type for the workers.
        dag_bucket: The GCS bucket holding the DAG files.
        log_bucket: The GCS bucket holding the task logs.
        service_account: The identity the workers run as.
        network: The VPC the web server sits in.
        subnetwork: The subnet the web server sits in.
        oauth_scope: The OAuth scope the workers request.
        python_version: The Python version in the Airflow environment.
        env_vars: The Airflow configuration, as a mapping.
        variables: The Airflow Variables to create.
        connections: The Airflow Connections to create.
        labels: The labels applied to the environment's resources.
        log_retention_days: How long task logs are kept.
        dag_retention_days: How long DAG files are kept in the bucket.
        bucket_regions: The region each bucket lives in, by bucket URI.
    """

    name: str
    location: str
    generation: str = "composer-2"
    size: str = "SMALL"
    machine_type: str = "n1-standard-2"
    dag_bucket: str = ""
    log_bucket: str = ""
    service_account: str = ""
    network: str = ""
    subnetwork: str = ""
    oauth_scope: str = "cloud-platform"
    python_version: str = "3.11"
    env_vars: dict[str, str] = field(default_factory=dict)
    variables: dict[str, str] = field(default_factory=dict)
    connections: dict[str, dict[str, str]] = field(default_factory=dict)
    labels: dict[str, str] = field(default_factory=dict)
    log_retention_days: int = 30
    dag_retention_days: int = 30
    bucket_regions: dict[str, str] = field(default_factory=dict)

    # -- derived properties ------------------------------------------

    @property
    def version_info(self) -> dict[str, Any]:
        """Return the version and executor for this generation.

        Returns:
            A dict with the executor, the Airflow version, the image
            version and the task startup cost.
        """
        return COMPOSER_GENERATIONS[self.generation]

    @property
    def executor(self) -> str:
        """Return the executor this environment uses.

        Returns:
            The executor's name.
        """
        return self.version_info["executor"]

    @property
    def airflow_version(self) -> str:
        """Return the pinned Airflow version.

        Returns:
            The Airflow version string.
        """
        return self.version_info["airflow_version"]

    @property
    def workers_min(self) -> int:
        """Return the minimum worker count.

        Returns:
            The minimum number of workers.
        """
        return ENVIRONMENT_SIZES[self.size]["workers_min"]

    @property
    def workers_max(self) -> int:
        """Return the maximum worker count.

        Returns:
            The maximum number of workers.
        """
        return ENVIRONMENT_SIZES[self.size]["workers_max"]

    @property
    def vcpu_per_worker(self) -> int:
        """Return the vCPUs per worker, derived from the machine type.

        Returns:
            The number of vCPUs.
        """
        return {"n1-standard-2": 2, "n1-standard-4": 4,
                "n1-standard-8": 8, "n2-standard-2": 2}[self.machine_type]

    @property
    def parallelism(self) -> int:
        """Return the recommended Airflow parallelism for this size.

        Returns:
            The parallelism setting.
        """
        return {"SMALL": 8, "MEDIUM": 32, "LARGE": 128}[self.size]

    # -- rendering ----------------------------------------------------

    def create_command(self) -> str:
        """Render the `gcloud composer environments create` command.

        Returns:
            The command, as a shell string.
        """
        parts = [
            "gcloud composer environments create", self.name,
            f"--location={self.location}",
            f"--image-version={self.version_info['image_version']}",
            f"--environment-size=ENVIRONMENT_SIZE_{self.size}",
            f"--machine-type={self.machine_type}",
            f"--dag-bucket={self.dag_bucket}",
            f"--log-bucket={self.log_bucket}",
            f"--oauth-scope={self.oauth_scope}",
            f"--python-version={self.python_version}",
        ]
        if self.service_account:
            parts.append(f"--service-account={self.service_account}")
        if self.network:
            parts.append(f"--web-server-network={self.network}")
        if self.subnetwork:
            parts.append(f"--web-server-subnetwork={self.subnetwork}")
        if self.env_vars:
            env = ",".join(f"{k}={v}" for k, v in sorted(self.env_vars.items()))
            parts.append(f"--env-vars={env}")
        if self.labels:
            labels = ",".join(f"{k}={v}" for k, v in sorted(self.labels.items()))
            parts.append(f"--labels={labels}")
        # The first element is the subcommand; the rest are flags and
        # values that contain no shell metacharacters worth quoting.
        return " \\\n  ".join(parts)

    def set_variable_commands(self) -> list[str]:
        """Render the commands that create the Airflow Variables.

        Returns:
            A list of shell commands.
        """
        return [
            f"gcloud composer environments update {self.name} "
            f"--location={self.location} "
            f"--update-airflow-variables={name}={value}"
            for name, value in sorted(self.variables.items())
        ]

    def describe(self) -> str:
        """Return a human-readable description of the environment.

        Returns:
            A multi-line description.
        """
        lines = [
            f"Composer environment: {self.name}",
            f"  location        : {self.location}",
            f"  generation      : {self.generation}",
            f"  executor        : {self.executor}",
            f"  Airflow version : {self.airflow_version}",
            f"  size            : {self.size} "
            f"({self.workers_min}-{self.workers_max} workers)",
            f"  machine type    : {self.machine_type} "
            f"({self.vcpu_per_worker} vCPU/worker)",
            f"  parallelism     : {self.parallelism}",
            f"  dag bucket      : {self.dag_bucket}",
            f"  log bucket      : {self.log_bucket}",
            f"  service account : {self.service_account or '(default)'}",
            f"  network         : {self.network or '(managed VPC)'}",
            f"  oauth scope     : {self.oauth_scope}",
            f"  python version  : {self.python_version}",
            f"  variables       : {len(self.variables)}",
            f"  connections     : {len(self.connections)}",
            f"  log retention   : {self.log_retention_days} days",
        ]
        return "\n".join(lines)

    # -- validation ---------------------------------------------------

    def validate(self) -> list[str]:
        """Check the environment for the mistakes that cost money.

        Returns:
            A list of problems; empty means the environment is valid.
        """
        problems = []

        if self.generation not in COMPOSER_GENERATIONS:
            problems.append(
                f"unknown generation {self.generation!r} "
                f"(expected one of {sorted(COMPOSER_GENERATIONS)})")
        if self.size not in ENVIRONMENT_SIZES:
            problems.append(
                f"unknown size {self.size!r} "
                f"(expected one of {sorted(ENVIRONMENT_SIZES)})")

        # The DAG bucket and the environment should be in the same region.
        if self.dag_bucket and not self.dag_bucket.startswith("gs://"):
            problems.append(f"dag_bucket must be a gs:// URI, got {self.dag_bucket!r}")
        if self.log_bucket and not self.log_bucket.startswith("gs://"):
            problems.append(f"log_bucket must be a gs:// URI, got {self.log_bucket!r}")
        if self.dag_bucket and self.log_bucket:
            if self.dag_bucket == self.log_bucket:
                problems.append(
                    "the DAG bucket and the log bucket must be different — "
                    "the log lifecycle would delete your DAG files")

        # A bucket in another region makes every DAG file read cross a
        # region boundary, and moves your data out of its residency region.
        for bucket, region in self.bucket_regions.items():
            if region != self.location:
                problems.append(
                    f"bucket {bucket!r} is in region {region!r} but the "
                    f"environment is in {self.location!r} — DAG file reads "
                    f"would cross a region boundary")

        # The web server must be in a VPC if you name a network.
        if bool(self.network) != bool(self.subnetwork):
            problems.append(
                "network and subnetwork must be set together — Composer "
                "requires both for a private web server")

        # The service account needs an email-like value.
        if self.service_account and "@" not in self.service_account:
            problems.append(
                f"service_account must be a full email address, "
                f"got {self.service_account!r}")

        # Log retention must be positive and finite.
        if self.log_retention_days <= 0:
            problems.append("log_retention_days must be positive")
        if self.log_retention_days > 365:
            problems.append(
                f"log_retention_days of {self.log_retention_days} is over a "
                f"year — check the storage cost")

        # The timezone must be explicit.
        if "AIRFLOW__CORE__DEFAULT_TIMEZONE" not in self.env_vars:
            problems.append(
                "AIRFLOW__CORE__DEFAULT_TIMEZONE is not set — set it "
                "explicitly to UTC so the scheduler and the warehouse agree")

        # A Composer 3 environment on a chatty DAG pays the pod tax.
        if self.generation == "composer-3" and self.size == "SMALL":
            problems.append(
                "Composer 3 with a SMALL environment has 1-2 workers, each "
                "paying a ~45s pod startup — check the task durations")

        return problems


# --------------------------------------------------------------------------
# A well-formed environment
# --------------------------------------------------------------------------
ENV = ComposerEnvironment(
    name="composer-prod",
    location="europe-west1",
    generation="composer-2",
    size="MEDIUM",
    machine_type="n1-standard-2",
    dag_bucket="gs://YOUR_PROJECT_ID-dags",
    log_bucket="gs://YOUR_PROJECT_ID-logs",
    service_account="composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com",
    network="YOUR_VPC_NETWORK",
    subnetwork="YOUR_VPC_SUBNET",
    oauth_scope="cloud-platform",
    python_version="3.11",
    env_vars={
        "AIRFLOW__CORE__DEFAULT_TIMEZONE": "UTC",
        "AIRFLOW__CORE__PARALLELISM": "32",
        "AIRFLOW__CORE__MAX_ACTIVE_TASKS_PER_DAG": "16",
        "AIRFLOW__WEBSERVER__EXPOSE_CONFIG": "True",
    },
    variables={
        "gcp_project_id": "YOUR_PROJECT_ID",
        "source_bucket": "YOUR_PROJECT_ID-source",
        "target_bucket": "YOUR_PROJECT_ID-target",
        "bq_dataset": "analytics",
        "bq_table": "orders_enriched",
        "full_load_threshold": "1000000",
    },
    connections={
        "google_cloud_default": {
            "conn_type": "google_cloud_platform",
            "extra": '{"project_id": "YOUR_PROJECT_ID"}',
        },
        "http_orders_api": {
            "conn_type": "http",
            "host": "api.example.com",
            "login": "svc_account",
            "password": "<stored in Secret Manager, not here>",
        },
    },
    labels={"team": "data-platform", "env": "prod"},
    log_retention_days=30,
    dag_retention_days=30,
    bucket_regions={
        "gs://YOUR_PROJECT_ID-dags": "europe-west1",
        "gs://YOUR_PROJECT_ID-logs": "europe-west1",
    },
)

print(ENV.describe())
print()

problems = ENV.validate()
print("Validation:", problems or "no problems")
print()

print("=" * 78)
print("The command you would run (run in a terminal, not executed here)")
print("=" * 78)
print(ENV.create_command())
print()

print("=" * 78)
print("Setting the Variables after creation")
print("=" * 78)
for command in ENV.set_variable_commands():
    print(command)

In [ ]:
"""The validator, exercised against environments that are wrong.

Each of these is a mistake that costs real money or real debugging time,
and every one of them is caught *before* the environment is created.
"""

from dataclasses import replace


def show(title, env):
    """Validate an environment and print the problems.

    Args:
        title: A short description of the scenario.
        env: The environment to validate.
    """
    print(f"--- {title} " + "-" * max(0, 60 - len(title)))
    problems = env.validate()
    if not problems:
        print("    no problems")
    for problem in problems:
        print(f"    ! {problem}")
    print()


# ---------------------------------------------------------------------
# 1. The DAG bucket and the log bucket are the same bucket
# ---------------------------------------------------------------------
show("the same bucket for DAGs and logs", replace(
    ENV,
    log_bucket="gs://YOUR_PROJECT_ID-dags",
))

# ---------------------------------------------------------------------
# 2. A bucket in another region
# ---------------------------------------------------------------------
show("the log bucket in another region", replace(
    ENV,
    bucket_regions={
        "gs://YOUR_PROJECT_ID-dags": "europe-west1",
        "gs://YOUR_PROJECT_ID-logs": "us-central1",
    },
))

# ---------------------------------------------------------------------
# 3. A private web server with no subnet
# ---------------------------------------------------------------------
show("a network with no subnetwork", replace(
    ENV,
    subnetwork="",
))

# ---------------------------------------------------------------------
# 4. No explicit timezone
# ---------------------------------------------------------------------
no_tz_vars = {k: v for k, v in ENV.env_vars.items()
              if k != "AIRFLOW__CORE__DEFAULT_TIMEZONE"}
show("no AIRFLOW__CORE__DEFAULT_TIMEZONE", replace(
    ENV,
    env_vars=no_tz_vars,
))

# ---------------------------------------------------------------------
# 5. Log retention set to a year
# ---------------------------------------------------------------------
show("a year of log retention", replace(
    ENV,
    log_retention_days=365,
))

# ---------------------------------------------------------------------
# 6. Composer 3 on a SMALL environment
# ---------------------------------------------------------------------
show("Composer 3 with a SMALL environment", replace(
    ENV,
    generation="composer-3",
    size="SMALL",
))

# ---------------------------------------------------------------------
# 7. A service account that is not an email address
# ---------------------------------------------------------------------
show("a service account that is not an email", replace(
    ENV,
    service_account="composer-sa",
))

print("=" * 78)
print("What the validator is for")
print("=" * 78)
print()
print("Every one of the seven problems above is:")
print()
print("  * invisible in the `gcloud` command's own validation,")
print("  * a real cost or a real outage once the environment exists, and")
print("  * free to catch before you create anything.")
print()
print("This is the argument for describing an environment as data. A")
print("description can be reviewed, diffed, tested and rendered into a")
print("command. A YAML file you typed by hand can only be eyeballed.")

# ---------------------------------------------------------------------
# The comparison table, from the model
# ---------------------------------------------------------------------
print()
print("=" * 78)
print("Composer 2 vs Composer 3, computed from the model")
print("=" * 78)
print()

for generation in ("composer-2", "composer-3"):
    env = replace(ENV, generation=generation, size="MEDIUM")
    info = env.version_info
    print(f"  {generation}")
    print(f"    executor         : {info['executor']}")
    print(f"    Airflow version  : {info['airflow_version']}")
    print(f"    image version    : {info['image_version']}")
    print(f"    task startup     : {info['task_startup_seconds']}s")
    print(f"    workers          : {env.workers_min}-{env.workers_max}")
    print(f"    parallelism      : {env.parallelism}")
    print()

# --- the startup tax, measured against the course's DAGs ---------------
print("=" * 78)
print("THE STARTUP TAX, against the course's DAGs")
print("=" * 78)
print()

from local_runner import DAGS  # noqa: E402

POD_STARTUP = COMPOSER_GENERATIONS["composer-3"]["task_startup_seconds"]
CELERY_DISPATCH = COMPOSER_GENERATIONS["composer-2"]["task_startup_seconds"]
# The course's pure compute, measured in milliseconds.
TASK_WORK_SECONDS = 0.02

print(f"{'dag':<18} {'tasks':>6} {'celery wall':>12} {'k8s wall':>10} "
      f"{'k8s overhead':>13}")
print("-" * 72)
for name in sorted(DAGS):
    dag = DAGS[name]()
    n = len(dag.tasks)
    celery = n * (CELERY_DISPATCH + TASK_WORK_SECONDS)
    k8s = n * (POD_STARTUP + TASK_WORK_SECONDS)
    overhead = (k8s - celery) / celery * 100
    print(f"{name:<18} {n:>6} {celery:>11.2f}s {k8s:>9.2f}s "
          f"{overhead:>12.0f}%")

print()
print("Every course DAG is a chain — at most one task is ever ready at a")
print("time — so the pod startup is paid *sequentially*.")
print()
print("The KubernetesExecutor is right for heavy tasks and wrong for")
print("chatty ones. These DAGs are the latter.")
print()

# --- the crossover ----------------------------------------------------
print("Crossover: the task duration at which the 45s pod tax is < 10% "
      "of the total.")
crossover = POD_STARTUP / 0.10 - POD_STARTUP
print(f"  pod startup {POD_STARTUP}s -> crossover at "
      f"{crossover / 60:.1f} minutes of task duration")
print()

# --- the environment's own startup cost --------------------------------
print("=" * 78)
print("What the choice costs the ENVIRONMENT, per task")
print("=" * 78)
print()
for generation in ("composer-2", "composer-3"):
    env = replace(ENV, generation=generation)
    startup = env.version_info["task_startup_seconds"]
    print(f"  {generation:<12} {env.executor:<20} "
          f"{startup:>3}s per task, "
          f"{env.workers_min}-{env.workers_max} workers")
print()
print("Rule: use Composer 3 when tasks need their own resources or their")
print("own images AND take more than ~8 minutes. Otherwise stay on 2.")

## 5. The Composer UI tour

The Composer web server is the standard Airflow UI, with a few Composer-specific additions. Six panes cover almost everything you will do.

### The DAGs list — your home screen

The landing page. Every DAG the scheduler has parsed, one row each:

| Column | What it tells you |
|---|---|
| **DAG** | The `dag_id` |
| **Schedule** | The cron expression, or "Dataset" if the DAG is dataset-scheduled, or "—" for a manual-only DAG |
| **Last Run** | When it last ran, and **how long it took** |
| **Next Run** | When it will run next — greyed out if the DAG is paused |
| **Age** | How long since the last run |
| **Owner** | From `default_args["owner"]` |
| **Latest run** | The state of the most recent run — green, yellow, red |

The rows also show a small coloured bar per task, so you can see *which* task failed without clicking into anything.

**A DAG missing from this list is the single most common Composer problem.** The DAG file did not parse. The scheduler's import-error log has the traceback.

```bash
# RUN IN A TERMINAL — not executed here.
# The DAGs that parsed successfully.
gcloud composer environments run list YOUR_ENVIRONMENT_NAME \
  --location=REGION
```

### The Grid / Calendar views

The **Grid** view is a calendar of runs: rows are DAGs, columns are dates, and each cell is coloured by the run's state. This is the view for answering "did yesterday's run happen?" and "was there a gap last Tuesday?".

The **Calendar** view (in newer Airflow versions, part of the Grid) shows the same data with more detail per cell — the run's duration, and the tasks inside it.

The practical use: **a wall of yellow and red cells is a capacity problem, not a data problem.** If every DAG in the same column is red, the workers were saturated and the run timed out waiting for a slot.

### The Graph view — the shape of the DAG

The Graph view renders the DAG as a node diagram. This is where you *see* the patterns from notebook 08:

- A **diamond** is a branch; the highlighted path is what the last run took.
- A **fan-out** with four parallel nodes is either a static fan-out or a dynamic mapping (the mapped instances show as `load_region[0]`, `[1]`, …).
- A **grey node** is a skipped task — usually a branch that was not taken.
- A **red node** is a failed task; click it to jump to its log.

**What to look for:** a task with many grey upstream nodes and no green path. That is the "join with the wrong trigger rule" bug from notebook 08, visible at a glance.

### The Task / Logs pane

Click any task instance and you get:

- **The task log** — the full stdout/stderr of the task, plus Airflow's own framing.
- **Rendered templates** — every template field, *before* and *after* rendering. This is how you debug `{{ var.value.bucket }}` that rendered to an empty string.
- **The try number** — which attempt this was, and the history of previous attempts.
- **XCom** — the values this task pushed and pulled.
- **Metadata** — the task's configuration, its dependencies, its trigger rule, its timeout, its retries.

**The log is where you actually debug.** The "Rendered Templates" tab is the most under-used feature in Airflow: when a path is wrong, the rendered value tells you in one glance what the variable actually resolved to.

**Composer-specific:** the logs also live in the **Cloud Storage logs bucket**. That matters when the environment itself is unhealthy — you can read a task's log with `gsutil` even if the web server is down.

```bash
# RUN IN A TERMINAL — not executed here.
# Read a task log straight from the bucket, with no UI.
gsutil ls gs://YOUR_PROJECT_ID-logs/
gsutil cat "gs://YOUR_PROJECT_ID-logs/gs://YOUR_PROJECT_ID-dags/etl_orders_to_bigquery/2026-01-15T00:00:00+00:00/load_to_bigquery/1.log"
```

### Admin → Variables and Connections

The environment-level configuration.

**Variables** — non-secret configuration that differs per *environment*: bucket names, project IDs, a threshold, an owner email. Set them here and the DAG reads `{{ var.value.bucket }}`.

**Connections** — credentials and connection details: a `conn_type`, a host, a login, a password, and a JSON `extra` blob. The DAG reads `{{ conn.my_conn.host }}`.

> **Never put a password in a Variable.** Variables are visible in the UI to everyone with Variable read access, and they are stored in the metadata database in plain text. Use a Connection — and, better, point the Connection's `extra` at Secret Manager.

### Admin → Config — the effective `airflow.cfg`

The environment's full configuration, rendered. When a DAG behaves differently in Composer than locally, this is where the difference is: a different `executor`, a different `default_timezone`, a different `max_map_length`, a different `sql_alchemy_pool_size`.

**Read it once, deliberately.** It answers most of "why does it work here and not there?"

### Admin → Datasets

The dataset update history, when the environment uses Datasets (Airflow 2.4+). Each URI, its last update time, and which DAGs consume it.

This pane is also where you confirm the gap from notebook 08: **if a producer stops updating its dataset, this view shows the last update time going stale — and nothing else will tell you.** That is why a monitoring check is needed.

### Admin → Audit Log

Who changed what, when: environment configuration changes, Variable and Connection edits, DAG runs triggered manually.

**This is the pane you read when something changed and nobody knows who.** For a production environment, enable Data Access logs and route them to a sink you control.

## 6. IAM — who the environment runs as

Composer is not magic. Every pod, every task, every GCS read and every BigQuery job runs as a **service account** — a Google Cloud identity with explicit roles.

The rule that governs everything else:

> **The environment's service account should be able to do exactly what the DAGs need, and nothing more.** A Composer environment whose service account is `Owner` is a production incident waiting for a bad DAG.

### The default service account, and why you must not keep it

If you omit `--service-account`, Composer creates a default identity named after your project:

```
YOUR_PROJECT_ID-composer@YOUR_PROJECT_ID.iam.gserviceaccount.com
```

It exists, and it works, and on day one you will not think about it again. Then one of these happens:

- A DAG task imports a library that reads a broad credential set.
- A `gsutil cp -r gs://YOUR_PROJECT_ID-*` in a task copies something it should not.
- Someone adds a `subprocess` call to a task and the pod inherits the whole identity.

The default account is also shared: it is the same identity for every environment in the project, so dev and prod share blast radius.

**Create a dedicated service account per environment, at least for prod.**

```bash
# RUN IN A TERMINAL — not executed here.
# 1. A dedicated identity for the environment.
gcloud iam service-accounts create composer-sa \
  --project=YOUR_PROJECT_ID \
  --display-name="Composer prod environment"

# 2. Grant the roles the tasks actually need.
gcloud projects add-iam-policy-binding YOUR_PROJECT_ID \
  --member="serviceAccount:composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/composer.worker"

gcloud projects add-iam-policy-binding YOUR_PROJECT_ID \
  --member="serviceAccount:composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/bigquery.jobUser"

gcloud projects add-iam-policy-binding YOUR_PROJECT_ID \
  --member="serviceAccount:composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/storage.objectAdmin"

gcloud projects add-iam-policy-binding YOUR_PROJECT_ID \
  --member="serviceAccount:composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/cloudresourcemanager.projectViewer"

# 3. Optionally, grant access to the secret holding the API password.
gcloud secrets add-iam-policy-binding YOUR_SECRET_ID \
  --member="serviceAccount:composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/secretmanager.secretAccessor"
```

Then pass it at creation time:

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --service-account=composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com
```

### The roles, and what each one is actually for

| Role | What it permits | Why the DAG needs it |
|---|---|---|
| `roles/composer.worker` | Talk to the GKE cluster, read the DAG bucket, write task logs | Without it, the worker pods cannot start |
| `roles/bigquery.jobUser` | Run BigQuery jobs; read datasets | The `load_to_bigquery` and `aggregate_by_region` tasks |
| `roles/storage.objectAdmin` | Read and write objects in the buckets | Reading raw CSVs, writing the export |
| `roles/cloudresourcemanager.projectViewer` | Read project metadata (project number, region) | The GCPHook resolves the project without an explicit `project_id` |
| `roles/secretmanager.secretAccessor` | Read one specific secret | Only if a Connection's `extra` points at Secret Manager |

`roles/composer.worker` is required — it is the one role Composer will not start without.

### Two ways to give a task a credential

**Option A — the default connection.** Composer creates `google_cloud_default` pointing at the environment's service account. The `GCPHook` and `BigQueryHook` pick it up automatically, with no code change.

**Option B — a named connection with a specific identity.** When one task needs a *different*, narrower identity than the rest:

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --update-airflow-connections=bigquery_reader;conn_type=google_cloud_platform;extra='{"project_id": "YOUR_PROJECT_ID", "key_path": "/opt/airflow/keys/reader.json"}'
```

**Option C — a federated connection.** Modern and clean: the service account is listed in the connection's `extra` as `{"extra__google_cloud_platform__scope": "https://www.googleapis.com/auth/cloud-platform"}` and the impersonation chain is resolved at call time. No key files, no key rotation.

### The secret in the connection's `extra`

The Connection stores its credentials in a JSON `extra` blob. For anything sensitive, point that blob at Secret Manager rather than embedding the value:

```json
{
  "conn_type": "http",
  "host": "api.example.com",
  "login": "svc_account",
  "password": "",
  "extra": {
    "extra__http__auth": "aws-iam-role",
    "secret_name": "orders-api-password"
  }
}
```

The password lives in Secret Manager, is versioned, is auditable, and is rotatable without touching the DAG.

> **Three rules, and they are the whole of IAM hygiene.**
>
> 1. **Never a password in a Variable.** Variables are visible in the UI to anyone with Variable read access and are stored in plain text in the metadata database.
> 2. **Never `Owner` on the environment service account.** `roles/composer.worker` plus exactly the data-plane roles the DAGs need.
> 3. **Never a downloaded key file.** Use the attached service account, or impersonation, or Secret Manager.

### Auditing what the environment can actually do

You do not need to guess. Ask IAM:

```bash
# RUN IN A TERMINAL — not executed here.
# What does the environment's identity have?
gcloud projects get-iam-policy YOUR_PROJECT_ID \
  --flatten="bindings[].members" \
  --filter="bindings.members:composer-sa@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --format="table(bindings.role)"

# Who else can change the environment's IAM?
gcloud projects get-iam-policy YOUR_PROJECT_ID \
  --flatten="bindings[].members" \
  --filter="bindings.role:roles/composer.user" \
  --format="table(bindings.members)"
```

The second command is the more interesting one. `roles/composer.user` is what a human needs to open the UI and trigger runs. Grant it to the handful of people who operate the pipeline — and review it periodically, because `composer.user` can trigger arbitrary DAG code, which means it is effectively code-execution on your data plane.

## 7. Cleanup and cost control

This is the section that separates a course project from a production platform. A Composer environment that nobody deletes bills every month it exists, whether or not a single DAG runs.

### What deleting an environment actually does

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments delete YOUR_ENVIRONMENT_NAME \
  --location=REGION
```

Deletion is asynchronous and takes several minutes. While it runs, the environment is in a `DELETING` state.

**What is destroyed:**

- The GKE cluster and all its nodes — the workers go with it.
- The Cloud SQL instance backing the Airflow metadata database — **and with it every DAG run, every task instance, every XCom, every Variable and every Connection**.
- The Composer-managed service account, if Composer created it.
- The Airflow web server, scheduler and their configuration.

**What survives:**

- **The DAG bucket.** The GCS bucket you pointed `--dag-bucket` at is yours; Composer never owned it.
- **The log bucket.** Same — your bucket, your data, your lifecycle rules.
- **The Cloud Storage objects the DAGs wrote.** Your data lake is not a Composer resource.
- **The BigQuery tables.** Ditto.
- **The IAM bindings.** Deleting the environment does not remove the roles you granted to the service account — but the account itself may be deleted, which leaves the bindings dangling.

That asymmetry is the thing to internalise:

> **Deleting an environment deletes the *orchestration*. It never deletes the *data*.** Back up the metadata database if you care about the run history; ignore it if you do not.

### The story: the environment nobody deleted

A common failure, told as it usually happens.

A team builds a sandbox environment for a spike. The spike ends. Nobody deletes it, because there is no alert and deleting it means losing the Variables somebody spent an afternoon configuring. It is a `SMALL` environment — the cheapest one — so nobody thinks about it.

Eleven months later, the quarterly billing review finds **$2,160 of unexplained Composer charges** for a project that "has no production pipelines."

An environment bills on **uptime**, not on usage. The web server, the scheduler and the Cloud SQL instance are running at 03:00 on a Sunday whether or not any DAG is scheduled then. The workers scale down when idle, but the fixed components do not.

The fix is boring and must happen *before* the environment is created, not eleven months later:

- **A `DELETING` policy** — a tag on the environment (`expiry=2026-01-15`) and a scheduled job that deletes anything past its date.
- **Budget alerts** on the project, with a threshold low enough to be acted on.
- **One environment per team, not per project**, so the fixed cost is paid once.

### Budget alerts, and why a threshold is not a limit

A budget alert sends a message. It does not stop anything. Set the threshold at roughly **50% of the expected monthly spend** so you get a warning with time to act.

```bash
# RUN IN A TERMINAL — not executed here.
gcloud billing budgets create \
  --billing-account=YOUR_BILLING_ACCOUNT_ID \
  --display-name="Composer prod budget" \
  --budget-amount=1000 \
  --budget-display-name="Composer prod budget" \
  --threshold-rule=0.5,0.9,1.0 \
  --threshold-type=PERCENT \
  --notifications-without-threshold
```

A budget can be scoped with `--filter`, e.g. `labels.env:prod`, so one alert per team does not drown in the noise of the whole project.

For a hard stop, you need something the budget cannot do: a Cloud Function on a Pub/Sub topic fed by the billing export, which deletes (or downgrades) an environment that crosses a hard threshold. Rare, but it is the only thing that actually stops a runaway.

### The bucket lifecycle rules — the cheapest cost control there is

The log bucket grows without bound. Task logs, scheduler logs and web server logs all land there, and an environment that runs 200 DAGs a day writes a surprising number of gigabytes.

Put a lifecycle rule on it the day you create it:

```bash
# RUN IN A TERMINAL — not executed here.
# Keep a week of logs, then delete them.
gsutil lifecycle set - gs://YOUR_PROJECT_ID-logs <<'EOF'
{
  "rule": [
    {
      "action": {"type": "Delete"},
      "condition": {"age": 7, "matchesPrefix": ["gs://YOUR_PROJECT_ID-logs/"]}
    }
  ]
}
EOF
```

**But note the trap the validator in notebook 09 flags:** never point `--dag-bucket` and `--log-bucket` at the *same* bucket. If they are the same bucket and the lifecycle rule matches that prefix, the rule deletes your DAG files, and the environment goes blank — every DAG disappears from the UI, and the traceback in the scheduler log is a confusing `FileNotFoundError` on a path that definitely exists.

That is precisely the failure the validator catches before anything is created:

```
--- the same bucket for DAGs and logs ------------------------------
    ! the DAG bucket and the log bucket must be different — the log
      lifecycle would delete your DAG files
```

### Retention, in one table

| Resource | Deleted with the environment? | Safe to delete manually? | Use a lifecycle rule? |
|---|---|---|---|
| GKE cluster | Yes | — | — |
| Cloud SQL metadata DB | Yes | Yes, if you have a backup | Yes, snapshot first |
| DAG bucket | **No** | Yes, if the DAGs live in Git | No — it is your source |
| Log bucket | **No** | Yes | **Yes — 7 to 30 days** |
| BigQuery tables | **No** | Depends on the dataset | Yes — partitioning + expiry |
| Service account | Sometimes | Yes | — |
| IAM bindings | **No** | Yes, prune them | — |
| VPC, subnets, Cloud NAT | **No** | Yes — these are project resources | No |

### A teardown checklist

The order matters: the environment must go first, because the IAM bindings reference its service account.

1. **Export anything you care about** — the DAGs are in Git; the run history usually is not.
2. **Delete the environment.**
3. **Wait for the state to reach `DELETED`.** Do not start step 4 while it is `DELETING`; the Cloud SQL instance is still billing.
4. **Delete the buckets** — the log bucket certainly; the DAG bucket only if the DAGs live in Git.
5. **Delete the service account** — and remove its IAM bindings, or they linger as dead members.
6. **Check the region.** An environment in `europe-west1` and its Cloud SQL instance are billed in `europe-west1`. Deleting a *different* region's environment does not touch the first one.
7. **Verify the bill.** The next day's usage report is the only proof.

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments list --location=REGION
gcloud composer environments describe YOUR_ENVIRONMENT_NAME --location=REGION
```

The last command's `state` field tells you whether you are done.

### The habits, summarised

1. **One environment per team**, sized to the peak concurrency, not to the number of DAGs.
2. **A lifecycle rule on the log bucket from day one.** It is free and it is the single biggest storage saving.
3. **A budget alert at 50%**, filtered by label so it is actionable.
4. **An expiry tag on every non-prod environment**, with a scheduled cleanup that honours it.
5. **Delete the environment before you delete the IAM bindings**, never after.
6. **Remember what survives.** The buckets and the tables are not Composer's — your data outlives the orchestration by design.

## 8. Your turn

Six exercises on Composer environments. Each one is a change you would actually make; each has a short answer.

### Exercise 1 — Spot the two environment descriptions that will cost you money

You inherit two environments. Read each and name the specific problem, and the specific fix.

**A.**

```python
ComposerEnvironment(
    name="analytics-dev",
    location="europe-west1",
    generation="composer-2",
    size="MEDIUM",
    dag_bucket="gs://acme-dev-dags",
    log_bucket="gs://acme-dev-logs",
    service_account="acme-dev@acme.iam.gserviceaccount.com",
    env_vars={"AIRFLOW__CORE__DEFAULT_TIMEZONE": "UTC"},
)
```

**B.**

```python
ComposerEnvironment(
    name="analytics-prod",
    location="us-central1",
    generation="composer-3",
    size="SMALL",
    dag_bucket="gs://acme-prod-logs",
    log_bucket="gs://acme-prod-logs",
    service_account="acme-prod@acme.iam.gserviceaccount.com",
    env_vars={},
)
```

### Exercise 2 — Choose the generation

Your DAGs have these task shapes:

- a 3-minute `PythonOperator` that reads a 40 MB CSV and writes a Parquet file
- a 25-minute `BigQueryOperator` that runs a heavy SQL query
- a 4-minute `PythonOperator` that calls a REST API and writes one row
- a 40-minute `SparkSubmitOperator` on a 32 GB dataset

Which Composer generation do you pick, and why? What is the crossover task duration you would use to justify it?

### Exercise 3 — Write the environment description

Write the `ComposerEnvironment` dataclass instance for a production environment with these requirements:

- location `europe-west1`, Composer 2, `MEDIUM`, `n1-standard-2`
- DAG bucket `gs://acme-prod-dags`, log bucket `gs://acme-prod-logs`, both in `europe-west1`
- service account `composer-prod@acme.iam.gserviceaccount.com`
- `AIRFLOW__CORE__DEFAULT_TIMEZONE=UTC`, `AIRFLOW__CORE__PARALLELISM=32`
- 30 days of log retention
- Variables: `gcp_project_id`, `source_bucket`, `bq_dataset`, `bq_table`
- 2-day DAG retention

Then call `.validate()` and `.create_command()` on it.

### Exercise 4 — The connection that should not be a Variable

A DAG reads an API password. Write it as:

1. a Connection with the password in `extra`
2. a Connection whose `extra` points at Secret Manager

Say which one you would ship and why, in one sentence each.

### Exercise 5 — Teardown order

You are decommissioning a `MEDIUM` environment in `europe-west1`. Its DAGs live in Git. Its logs bucket has 200 GB in it. Its service account has four project-level role bindings.

Write the teardown steps in the correct order, and state for each resource whether it survives the environment deletion.

### Exercise 6 — Budget the environment

A `MEDIUM` environment costs $340/month at full utilisation. The team wants three environments (dev, staging, prod). Their utilisation is 25%, 50% and 100% respectively.

Using the cost model from section 4:

1. What is the total monthly cost of the three environments together?
2. What would the total be if all three DAG sets ran in a single `MEDIUM` environment at 100% utilisation?
3. What is the annual saving from consolidating?

Explain, in one paragraph, why the answer is not the difference you might expect.

<details>
<summary><b>Solution (click to reveal)</b></summary>

<details>
<summary><strong>Solutions</strong></summary>

### Exercise 1

**A** has two problems the validator will catch, and one it will not.

The two it catches:

- **The service account is not scoped.** `acme-dev@acme.iam.gserviceaccount.com` is fine as a string (it has an `@`), but there is no dedicated account *and* no roles declared — a dev environment running as a shared identity with prod's blast radius. Fix: create `composer-dev@acme.iam.gserviceaccount.com` and grant `roles/composer.worker`, `roles/bigquery.jobUser`, `roles/storage.objectAdmin`, `roles/cloudresourcemanager.projectViewer`.
- **`log_retention_days` and `dag_retention_days` are defaults (30) but `bucket_regions` is empty**, so the validator cannot confirm the buckets are in `europe-west1`. Fix: declare `bucket_regions={"gs://acme-dev-dags": "europe-west1", "gs://acme-dev-logs": "europe-west1"}`.

The one it will not catch: **`MEDIUM` for a dev environment.** The cost model shows a `MEDIUM` environment costs $340/month whether it runs 25% of the time or 100%. A dev environment usually runs a handful of DAGs a day. Fix: `size="SMALL"` (1–2 workers), which cuts the fixed monthly cost from $340 to $180 and saves $160/month for a $0 material difference in capability. A dev environment does not need `AIRFLOW__CORE__PARALLELISM=32`.

**B** has three problems, and the first is the one that ends a career.

- **The DAG bucket and the log bucket are the same bucket.** The validator says it exactly:

  ```
  ! the DAG bucket and the log bucket must be different — the log
    lifecycle would delete your DAG files
  ```

  The moment you put a lifecycle rule on the logs (you will — see Exercise 5), the rule matches the shared prefix and Composer cannot find the DAG files. Every DAG vanishes from the UI and the scheduler log fills with `FileNotFoundError`. Fix: two buckets, always.

- **`env_vars={}` — no `AIRFLOW__CORE__DEFAULT_TIMEZONE`.** The validator catches it:

  ```
  ! AIRFLOW__CORE__DEFAULT_TIMEZONE is not set — set it explicitly to
    UTC so the scheduler and the warehouse agree
  ```

  Without it, the environment inherits the default timezone, and a daily ETL whose source data lands at 23:30 UTC will partition into the wrong day. Fix: `env_vars={"AIRFLOW__CORE__DEFAULT_TIMEZONE": "UTC"}`.

- **Composer 3 on a `SMALL` environment.** The validator catches it:

  ```
  ! Composer 3 with a SMALL environment has 1-2 workers, each paying a
    ~45s pod startup — check the task durations
  ```

  Each task pays ~45 s of pod startup. If the DAGs are chatty chains, that tax is paid sequentially, task after task. Fix: either Composer 2, or Composer 3 on `MEDIUM` with long-running tasks.

### Exercise 2

**Composer 3** — because of the Spark job.

The reasoning, task by task:

| Task | Duration | Verdict |
|---|---|---|
| 3-minute Parquet write | 3 min | Wrong for Composer 3 — the 45 s pod startup is 25% of the task's wall time |
| 25-minute BigQuery query | 25 min | Composer 3 is fine; a query this long benefits from its own resources |
| 4-minute REST call | 4 min | Wrong for Composer 3 — startup is 19% overhead |
| 40-minute Spark submit | 40 min | **Composer 3, clearly** — a Spark task wants its own node, its own image and its own memory |

A single environment runs one executor. If the Spark task needs a Kubernetes-native pod with a Spark image and 32 GB, the environment must be `composer-3`. The short tasks then pay the startup tax — but 45 s against a 3-minute task is an acceptable price for one environment that can run everything.

**The crossover:** the task duration at which the 45 s pod tax is below 10% of the total.

```
crossover = 45 / 0.10 - 45 = 405 seconds = 6.8 minutes
```

So: **roughly 7 minutes.** Below that, the pod startup dominates and Celery (Composer 2, ~1 s dispatch) is the better executor. Above it, the tax is noise.

And note the crossover is not a cliff — it is a *margin* argument. If you would accept a 25% overhead, the crossover drops to 60 s of task duration, and Composer 2 is almost never the wrong answer for a chatty DAG.

### Exercise 3

```python
PROD = ComposerEnvironment(
    name="analytics-prod",
    location="europe-west1",
    generation="composer-2",
    size="MEDIUM",
    machine_type="n1-standard-2",
    dag_bucket="gs://acme-prod-dags",
    log_bucket="gs://acme-prod-logs",
    service_account="composer-prod@acme.iam.gserviceaccount.com",
    oauth_scope="cloud-platform",
    python_version="3.11",
    env_vars={
        "AIRFLOW__CORE__DEFAULT_TIMEZONE": "UTC",
        "AIRFLOW__CORE__PARALLELISM": "32",
    },
    variables={
        "gcp_project_id": "acme",
        "source_bucket": "acme-source",
        "bq_dataset": "analytics",
        "bq_table": "orders_enriched",
    },
    log_retention_days=30,
    dag_retention_days=2,
    bucket_regions={
        "gs://acme-prod-dags": "europe-west1",
        "gs://acme-prod-logs": "europe-west1",
    },
)

print(PROD.validate())
print()
print(PROD.create_command())
```

`validate()` returns `[]` — no problems. The rendered command:

```
gcloud composer environments create analytics-prod \
  --location=europe-west1 \
  --image-version=composer-2.9.3-airflow-2.9.3 \
  --environment-size=ENVIRONMENT_SIZE_MEDIUM \
  --machine-type=n1-standard-2 \
  --dag-bucket=gs://acme-prod-dags \
  --log-bucket=gs://acme-prod-logs \
  --oauth-scope=cloud-platform \
  --python-version=3.11 \
  --service-account=composer-prod@acme.iam.gserviceaccount.com \
  --env-vars=AIRFLOW__CORE__DEFAULT_TIMEZONE=UTC,AIRFLOW__CORE__PARALLELISM=32
```

The Variables are set separately, after creation — `--update-airflow-variables` on `environments create` works but is easier to audit as its own step, and it is how you can re-apply them idempotently:

```
gcloud composer environments update analytics-prod --location=europe-west1 --update-airflow-variables=bq_dataset=analytics
gcloud composer environments update analytics-prod --location=europe-west1 --update-airflow-variables=bq_table=orders_enriched
gcloud composer environments update analytics-prod --location=europe-west1 --update-airflow-variables=gcp_project_id=acme
gcloud composer environments update analytics-prod --location=europe-west1 --update-airflow-variables=source_bucket=acme-source
```

Note the two-day DAG retention: the DAG files are in Git, so the bucket only needs to hold the current version. Thirty days of *log* retention is generous enough to debug a monthly report.

### Exercise 4

**1. Password in `extra`:**

```python
connections={
    "orders_api": {
        "conn_type": "http",
        "host": "api.example.com",
        "login": "svc_account",
        "password": "s3cr3t",
        "extra": "{}",
    },
}
```

**2. Pointing `extra` at Secret Manager:**

```python
connections={
    "orders_api": {
        "conn_type": "http",
        "host": "api.example.com",
        "login": "svc_account",
        "password": "",
        "extra": {
            "extra__http__auth": "aws-iam-role",
            "secret_name": "orders-api-password",
        },
    },
}
```

**Ship version 2.** Version 1 puts the password in the Airflow metadata database in plain text, visible in the UI to anyone with Connection read access, and versioned in the audit log every time it is edited.

Version 2 keeps the secret in Secret Manager — versioned, auditable, rotatable without touching the DAG, and revocable without touching the Airflow environment at all. The service account needs `roles/secretmanager.secretAccessor` on that one secret, and nothing else.

The general principle: **the Connection holds *where* the credential is; the secret store holds the credential.**

### Exercise 5

The teardown, in order:

**Step 1 — export what matters.** The DAGs are in Git, so nothing to export there. The run history lives in the Cloud SQL metadata database, which the next step destroys. If anyone will ever ask "did the January 15 run succeed?", take a snapshot:

```bash
gcloud sql export sql INSTANCES_ID gs://acme-backups/metadata-$(date +%F).sql --project=acme
```

**Step 2 — delete the environment.**

```bash
gcloud composer environments delete analytics-prod --location=europe-west1
```

This destroys the GKE cluster, the Cloud SQL instance (with all run history, XComs, Variables and Connections), the web server, the scheduler, and Composer-managed identities.

**Step 3 — wait for `DELETED`.** Not `DELETING`. While it is `DELETING`, the Cloud SQL instance is still billing.

```bash
gcloud composer environments describe analytics-prod --location=europe-west1 --format="value(state)"
```

**Step 4 — delete the log bucket.** 200 GB of logs, and no lifecycle rule means nobody will delete them for you.

```bash
gsutil rm -r gs://acme-prod-logs
```

**Step 5 — the DAG bucket.** The DAGs are in Git, so the bucket is disposable. But verify the sync is working *before* you delete it — a bucket with the DAGs and no Git history is the only copy:

```bash
gsutil ls gs://acme-prod-dags
```

**Step 6 — the service account and its bindings.** The bindings are project-level and do **not** die with the environment, so remove them explicitly, then delete the account:

```bash
gcloud projects remove-iam-policy-binding acme \
  --member="serviceAccount:composer-prod@acme.iam.gserviceaccount.com" \
  --role="roles/composer.worker"
# ... and the other three

gcloud iam service-accounts delete composer-prod@acme.iam.gserviceaccount.com
```

**Step 7 — the VPC.** The VPC, subnets and Cloud NAT are project resources and survive. If they were created for this environment alone, delete them too — but check first whether anything else uses them.

**Step 8 — verify the bill.** The next day's usage report is the only proof that the money stopped.

**Survival table:**

| Resource | Survives? |
|---|---|
| GKE cluster | No — destroyed |
| Cloud SQL metadata DB | No — destroyed (snapshot first if you need history) |
| Web server / scheduler | No — destroyed |
| DAG bucket | **Yes** — yours; delete manually if Git is the source of truth |
| Log bucket | **Yes** — yours; delete manually |
| BigQuery tables | **Yes** — not a Composer resource |
| IAM bindings | **Yes** — remove them explicitly |
| Service account | Sometimes — Composer deletes it only if it created it |
| VPC / subnets / Cloud NAT | **Yes** — project resources |

### Exercise 6

**1. Total monthly cost of the three environments.**

Using the model from section 4 (`monthly_cost(spec, utilisation=u)`), with the `MEDIUM` spec's worker midpoint of 4.5 workers:

| Environment | Utilisation | Workers | Total/month |
|---|---|---|---|
| dev | 25% | 1.125 | $186.25 |
| staging | 50% | 2.25 | $221.48 |
| prod | 100% | 4.5 | $340.00 |
| **Total** | | | **$747.73** |

**2. One `MEDIUM` environment at 100% utilisation.** **$340.00/month.**

**3. Annual saving from consolidating.** $(747.73 - 340.00) \times 12 = **$4,892.76/year.**

**Why it is not the difference you might expect.**

A naive calculation assumes the cost scales with usage: dev is cheap because it is used a quarter of the time, so merging should save three-quarters of dev's cost. It does not, because **the fixed components do not shrink with usage**.

For a `MEDIUM` environment, the fixed monthly cost is $340 — and the workers' variable compute is a *small* part of the bill. At 25% utilisation the bill is $186.25, so the fixed floor for a `MEDIUM` environment is roughly $175–180/month of pure "being switched on". The variable worker compute at 25% utilisation is only about $6 of that.

So the saving from deleting a `MEDIUM` environment is roughly **$186** — you eliminate the web server, the scheduler, the Cloud SQL instance and the GKE control plane, and you keep the small amount of worker compute by folding it into the surviving environment.

The deeper lesson is the same one from the utilisation table in section 4: **an idle environment is not free, and it is not cheap either.** The 25%-utilised dev environment costs 55% of a fully-utilised one, because the always-on half of the bill is fixed. That is why the rule is "one environment per team, sized for the peak" — not "one environment per project, per stage, per team".

And the caveat: consolidation only works if the DAGs are compatible. A dev environment that runs a broken DAG should not be able to break prod, and prod's data should not be visible to someone who is debugging. If you need that separation, you need separate environments, and you accept the $186/month floor as the price of the isolation.

</details>

</details>